# 03 — Baseline models

This notebook benchmarks three discrete-time baselines for PhysioNet 2012 mortality prediction:

- **Persistence MLP**: last value, mean and observation count per variable;
- **GRU**: forward-filled values, missingness mask and elapsed-time features;
- **GRU-D**: learned input and hidden-state decay driven by time since last observation.

All models use the same hourly representation and preprocessing. Model selection and the Event-1 threshold are determined on a stratified Set-A validation split. Final models are retrained on all Set A before retrospective evaluation on Sets B and C.

In [1]:
import sys, copy, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Start Jupyter from repository root or notebooks/.")

sys.path.insert(0, str(PROJECT_ROOT))

from src.data.physionet import load_split
from src.data.preprocessing import Physionet2012Dataset, build_vocab, fill_forward
from src.models.baselines import PersistenceBaseline, GRUBaseline, GRUD
from src.training.metrics import select_event1_threshold, challenge_metrics

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def reset_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

reset_seeds()

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

Project root: C:\Users\guido\git_projects\icu-continuous-forecasting
Device: cpu


## Development split

In [2]:
A = load_split(RAW_DIR, split="set-a")
assert len(A) == 4000
assert all(r.label is not None for r in A)

yA = np.asarray([int(r.label) for r in A], dtype=np.int64)
idx = np.arange(len(A))

tr_idx, va_idx = train_test_split(
    idx,
    test_size=0.20,
    random_state=SEED,
    stratify=yA,
)

A_tr = [A[i] for i in tr_idx]
A_va = [A[i] for i in va_idx]

dev_vocab = build_vocab(A_tr)

dev_tr_ds = Physionet2012Dataset(A_tr, dev_vocab)
dev_norm = dev_tr_ds.fit_normalizer()
dev_va_ds = Physionet2012Dataset(A_va, dev_vocab, normalizer=dev_norm)

print("A train/val:", len(A_tr), len(A_va))
print("Vocabulary:", len(dev_vocab))
print("Mortality:", round(yA[tr_idx].mean(), 4), round(yA[va_idx].mean(), 4))

A train/val: 3200 800
Vocabulary: 36
Mortality: 0.1384 0.1388


## Training

In [3]:
BATCH = 128
MAX_EPOCHS = 30
PATIENCE = 6
LR = 1e-3
GRAD_CLIP = 5.0

MODEL_NAMES = ["Persistence", "GRU", "GRU-D"]

def make_model(name, D, S):
    if name == "Persistence":
        return PersistenceBaseline(D, S)
    if name == "GRU":
        return GRUBaseline(D, S)
    if name == "GRU-D":
        return GRUD(D, S)
    raise ValueError(name)

def iter_batches(ds, batch_size=BATCH, shuffle=False, epoch=0):
    ids = np.arange(len(ds))
    if shuffle:
        rng = np.random.RandomState(SEED + int(epoch))
        rng.shuffle(ids)

    for start in range(0, len(ids), batch_size):
        yield ds.collate(ids[start:start + batch_size].tolist()).to(DEVICE)

def forward_model(name, model, batch):
    if name == "Persistence":
        return model(batch)

    if name == "GRU":
        filled = torch.from_numpy(
            fill_forward(
                batch.values.detach().cpu().numpy(),
                batch.mask.detach().cpu().numpy(),
            )
        ).to(DEVICE)
        return model(batch, filled)

    x_mean = torch.zeros(
        batch.values.shape[-1],
        dtype=batch.values.dtype,
        device=DEVICE,
    )
    return model(batch, x_mean)

def train_epoch(name, model, ds, optimizer, epoch):
    model.train()
    loss_fn = nn.BCEWithLogitsLoss()
    total = 0.0
    n = 0

    for batch in iter_batches(ds, shuffle=True, epoch=epoch):
        optimizer.zero_grad(set_to_none=True)
        logits = forward_model(name, model, batch)
        loss = loss_fn(logits, batch.labels.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        bs = len(batch.labels)
        total += float(loss.detach()) * bs
        n += bs

    return total / n

@torch.no_grad()
def predict(name, model, ds):
    model.eval()
    out = []

    for batch in iter_batches(ds, shuffle=False):
        logits = forward_model(name, model, batch)
        out.append(torch.sigmoid(logits).cpu().numpy())

    return np.concatenate(out)

def develop(name, train_ds, val_ds):
    D = len(train_ds.vocab)
    S = len(train_ds.static_names)

    reset_seeds()
    model = make_model(name, D, S).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    y_val = np.asarray([int(r.label) for r in val_ds.records])
    best_ap = -np.inf
    best_epoch = None
    best_state = None
    stale = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.perf_counter()
        loss = train_epoch(name, model, train_ds, optimizer, epoch)
        risk = predict(name, model, val_ds)

        auc = roc_auc_score(y_val, risk)
        ap = average_precision_score(y_val, risk)
        elapsed = time.perf_counter() - t0

        history.append({
            "epoch": epoch,
            "loss": loss,
            "val_AUROC": auc,
            "val_AUPRC": ap,
            "seconds": elapsed,
        })

        if ap > best_ap + 1e-5:
            best_ap = ap
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
            status = "[BEST]"
        else:
            stale += 1
            status = f"[patience {stale}/{PATIENCE}]"

        print(
            f"[{name}] {epoch:02d} loss={loss:.4f} "
            f"val_AUROC={auc:.4f} val_AUPRC={ap:.4f} "
            f"time={elapsed:.1f}s {status}"
        )

        if stale >= PATIENCE:
            break

    model.load_state_dict(best_state)
    risk = predict(name, model, val_ds)
    threshold = select_event1_threshold(y_val, risk)

    return model, best_epoch, threshold, pd.DataFrame(history)

In [4]:
dev_models = {}
selection_rows = []
histories = {}

for name in MODEL_NAMES:
    model, epochs, threshold, history = develop(name, dev_tr_ds, dev_va_ds)
    dev_models[name] = model
    histories[name] = history

    best = history.loc[history["epoch"] == epochs].iloc[0]

    selection_rows.append({
        "Model": name,
        "epochs": epochs,
        "threshold": threshold,
        "val_AUROC": best["val_AUROC"],
        "val_AUPRC": best["val_AUPRC"],
    })

selection = pd.DataFrame(selection_rows).set_index("Model")
display(selection.round(4))

[Persistence] 01 loss=0.3876 val_AUROC=0.7491 val_AUPRC=0.3560 time=3.8s [BEST]
[Persistence] 02 loss=0.3522 val_AUROC=0.7974 val_AUPRC=0.4183 time=3.8s [BEST]
[Persistence] 03 loss=0.3335 val_AUROC=0.8188 val_AUPRC=0.4624 time=3.6s [BEST]
[Persistence] 04 loss=0.3178 val_AUROC=0.8205 val_AUPRC=0.4553 time=3.7s [patience 1/6]
[Persistence] 05 loss=0.3133 val_AUROC=0.8473 val_AUPRC=0.5033 time=3.9s [BEST]
[Persistence] 06 loss=0.3044 val_AUROC=0.8507 val_AUPRC=0.5241 time=4.0s [BEST]
[Persistence] 07 loss=0.2959 val_AUROC=0.8518 val_AUPRC=0.5218 time=3.6s [patience 1/6]
[Persistence] 08 loss=0.2942 val_AUROC=0.8547 val_AUPRC=0.5325 time=3.6s [BEST]
[Persistence] 09 loss=0.2931 val_AUROC=0.8534 val_AUPRC=0.5139 time=3.6s [patience 1/6]
[Persistence] 10 loss=0.2832 val_AUROC=0.8568 val_AUPRC=0.5233 time=3.6s [patience 2/6]
[Persistence] 11 loss=0.2811 val_AUROC=0.8542 val_AUPRC=0.5110 time=3.6s [patience 3/6]
[Persistence] 12 loss=0.2879 val_AUROC=0.8522 val_AUPRC=0.5014 time=3.6s [patien

,epochs,threshold,val_AUROC,val_AUPRC
Model,,,,
Persistence,8,0.2551,0.8547,0.5325
GRU,11,0.2571,0.8701,0.5684
GRU-D,7,0.3300,0.8525,0.5305


## Final fit

In [5]:
final_vocab = build_vocab(A)

A_ds = Physionet2012Dataset(A, final_vocab)
final_norm = A_ds.fit_normalizer()

assert final_vocab == dev_vocab

D = len(final_vocab)
S = len(A_ds.static_names)

def train_fixed(name, ds, epochs):
    reset_seeds()
    model = make_model(name, D, S).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, int(epochs) + 1):
        t0 = time.perf_counter()
        loss = train_epoch(name, model, ds, optimizer, epoch)
        print(
            f"[{name}] {epoch:02d}/{int(epochs):02d} "
            f"loss={loss:.4f} time={time.perf_counter() - t0:.1f}s"
        )

    return model

final_models = {
    name: train_fixed(name, A_ds, selection.loc[name, "epochs"])
    for name in MODEL_NAMES
}

[Persistence] 01/08 loss=0.3903 time=5.9s
[Persistence] 02/08 loss=0.3469 time=5.6s
[Persistence] 03/08 loss=0.3237 time=3.6s
[Persistence] 04/08 loss=0.3124 time=3.5s
[Persistence] 05/08 loss=0.3144 time=3.4s
[Persistence] 06/08 loss=0.3017 time=3.4s
[Persistence] 07/08 loss=0.2969 time=3.4s
[Persistence] 08/08 loss=0.2970 time=3.7s
[GRU] 01/11 loss=0.4595 time=5.8s
[GRU] 02/11 loss=0.3399 time=5.4s
[GRU] 03/11 loss=0.3144 time=5.4s
[GRU] 04/11 loss=0.3023 time=5.3s
[GRU] 05/11 loss=0.2965 time=5.1s
[GRU] 06/11 loss=0.2889 time=5.2s
[GRU] 07/11 loss=0.2828 time=5.2s
[GRU] 08/11 loss=0.2782 time=5.1s
[GRU] 09/11 loss=0.2705 time=5.3s
[GRU] 10/11 loss=0.2672 time=5.6s
[GRU] 11/11 loss=0.2551 time=5.1s
[GRU-D] 01/07 loss=0.5188 time=6.7s
[GRU-D] 02/07 loss=0.3405 time=7.1s
[GRU-D] 03/07 loss=0.3122 time=6.9s
[GRU-D] 04/07 loss=0.2984 time=6.9s
[GRU-D] 05/07 loss=0.2896 time=6.9s
[GRU-D] 06/07 loss=0.2804 time=7.4s
[GRU-D] 07/07 loss=0.2756 time=6.7s


## Retrospective evaluation

In [6]:
B = load_split(RAW_DIR, split="set-b")
C = load_split(RAW_DIR, split="set-c")

assert len(B) == len(C) == 4000
assert all(r.label is None for r in B)
assert all(r.label is None for r in C)

B_ds = Physionet2012Dataset(B, final_vocab, normalizer=final_norm)
C_ds = Physionet2012Dataset(C, final_vocab, normalizer=final_norm)

risk_B = {
    name: predict(name, final_models[name], B_ds)
    for name in MODEL_NAMES
}
risk_C = {
    name: predict(name, final_models[name], C_ds)
    for name in MODEL_NAMES
}

print("PREDICTIONS FROZEN. B/C outcomes have not been read up to this point.")

PREDICTIONS FROZEN. B/C outcomes have not been read up to this point.


In [7]:
def read_outcomes(path):
    df = pd.read_csv(path, dtype={"RecordID": str})
    df["RecordID"] = df["RecordID"].str.strip()
    assert df["RecordID"].is_unique
    return df.set_index("RecordID")["In-hospital_death"]

def aligned_y(records, outcome_map):
    ids = [str(r.record_id).strip() for r in records]

    missing = sorted(set(ids) - set(outcome_map.index))
    extra = sorted(set(outcome_map.index) - set(ids))

    assert not missing, f"Missing outcome IDs: {missing[:10]}"
    assert not extra, f"Outcome IDs without records: {extra[:10]}"

    return np.asarray(
        [int(outcome_map.loc[rid]) for rid in ids],
        dtype=np.int64,
    )

yB = aligned_y(B, read_outcomes(RAW_DIR / "Outcomes-b.txt"))
yC = aligned_y(C, read_outcomes(RAW_DIR / "Outcomes-c.txt"))

rows = []

for name in MODEL_NAMES:
    threshold = float(selection.loc[name, "threshold"])

    rows.append({
        "Model": name,
        "Set": "B",
        **challenge_metrics(yB, risk_B[name], threshold),
    })
    rows.append({
        "Model": name,
        "Set": "C",
        **challenge_metrics(yC, risk_C[name], threshold),
    })

results = pd.DataFrame(rows).rename(columns={
    "auroc": "AUROC",
    "auprc": "AUPRC",
    "event1": "Event1",
    "sensitivity": "Sensitivity",
    "ppv": "PPV",
    "threshold": "Threshold",
    "event2": "Event2",
})

display(results.round(4))

,Model,Set,AUROC,AUPRC,Event1,Sensitivity,PPV,Threshold,Event2
0,Persistence,B,0.8393,0.5008,0.3715,0.3715,0.5582,0.2551,325.9455
1,Persistence,C,0.8396,0.4982,0.3641,0.3641,0.5741,0.2551,395.5216
2,GRU,B,0.8592,0.5347,0.5299,0.5299,0.5327,0.2571,119.2144
3,GRU,C,0.8547,0.5446,0.5197,0.5197,0.5232,0.2571,224.5535
4,GRU-D,B,0.8534,0.5311,0.4468,0.6285,0.4468,0.3300,88.2307
5,GRU-D,C,0.8538,0.5312,0.4640,0.6051,0.4640,0.3300,57.1411


## Summary

The comparison tests whether explicit recurrent modeling improves over a non-sequential summary baseline under identical preprocessing. AUROC/AUPRC measure discrimination, while Event 1 evaluates the frozen mortality decision threshold and Event 2 is sensitive to probability calibration.

In [8]:
c_results = (
    results[results["Set"] == "C"]
    .sort_values("AUROC", ascending=False)
    .reset_index(drop=True)
)

display(c_results.round(4))

assert len(results) == 6
assert set(results["Set"]) == {"B", "C"}
assert np.isfinite(results[["AUROC", "AUPRC", "Event1", "Event2"]].to_numpy()).all()

print("Notebook 03 completed successfully.")

,Model,Set,AUROC,AUPRC,Event1,Sensitivity,PPV,Threshold,Event2
0,GRU,C,0.8547,0.5446,0.5197,0.5197,0.5232,0.2571,224.5535
1,GRU-D,C,0.8538,0.5312,0.4640,0.6051,0.4640,0.3300,57.1411
2,Persistence,C,0.8396,0.4982,0.3641,0.3641,0.5741,0.2551,395.5216


Notebook 03 completed successfully.
